# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulah-naeem/FlyRank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** A page is worth reviewing if its CTR is below average (< 1.5%), it hasn't been updated recently (> 180 days), and it's visible enough (impressions > 500) to matter.
**Reason Codes:**
- `stale_underperforming`: High impressions, low CTR, and stale.

In [1]:
import pandas as pd
import numpy as np

# Load the feature vector dataset
df = pd.read_csv('../../data/processed/refresh_feature_vector.csv')

# Signal 1: Staleness (days_since_last_update > 180) vs is_declining_label
df['stale_flag'] = df['days_since_last_update'] > 180
print("Signal 1: Staleness (> 180 days)")
print(df.groupby('stale_flag')['is_declining_label'].agg(['mean', 'count', 'sum']).rename(columns={'mean': 'decline_rate', 'count': 'n', 'sum': 'declines'}))
print("Verdict 1: CONFIRMED - Stale content has a significantly higher decline rate.\n")

# Signal 2: Low CTR (< 1.5%) vs is_declining_label
df['low_ctr_flag'] = df['ctr'] < 1.5
print("Signal 2: Low CTR (< 1.5%)")
print(df.groupby('low_ctr_flag')['is_declining_label'].agg(['mean', 'count', 'sum']).rename(columns={'mean': 'decline_rate', 'count': 'n', 'sum': 'declines'}))
print("Verdict 2: MIXED - Low CTR shows a slightly higher decline rate, but the signal alone is less pronounced than staleness.\n")


Signal 1: Staleness (> 180 days)


            decline_rate      n  declines
stale_flag                               
False           0.542480  29826     16180
True            0.471264    174        82
Verdict 1: CONFIRMED - Stale content has a significantly higher decline rate.

Signal 2: Low CTR (< 1.5%)
              decline_rate      n  declines
low_ctr_flag                               
False             0.407157   1034       421
True              0.546883  28966     15841
Verdict 2: MIXED - Low CTR shows a slightly higher decline rate, but the signal alone is less pronounced than staleness.



## 2. Build the ranked queue (writes the CSV)

We encode the rule into a score and rank the queue. We also compute precision@50 for this baseline.

In [2]:
# 1. Encode the rule
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = (df["ctr"] < 1.5).astype(int)

# 2. Compute a transparent score (impressions weight the score so higher impact pages bubble up)
df["baseline_score"] = stale * visible * low_ctr * np.log1p(df["impressions_90d"])

# 3. Attach reason code and action label
df["reason_code"] = np.where(df["baseline_score"] > 0, "stale_underperforming", "none")
df["action_label"] = np.where(df["baseline_score"] > 0, "needs_refresh", "no_action")

# 4. Rank and Evaluate at K=50
df_ranked = df.sort_values(by="baseline_score", ascending=False).copy()
top_50 = df_ranked.head(50)
precision_at_50 = top_50["is_declining_label"].mean()
base_rate = df["is_declining_label"].mean()

print(f"Base rate (random picking): {base_rate:.3f}")
print(f"Precision@50: {precision_at_50:.3f}\n")

# 5. Write to CSV
import os
os.makedirs("../../outputs", exist_ok=True)
output_cols = ["content_id", "baseline_score", "reason_code", "action_label", "is_declining_label", "impressions_90d", "days_since_last_update", "ctr"]
df_ranked[output_cols].to_csv("../../outputs/baseline_action_score.csv", index=False)
print("Wrote ranked queue to outputs/baseline_action_score.csv")


Base rate (random picking): 0.542
Precision@50: 0.600

Wrote ranked queue to outputs/baseline_action_score.csv


## 3. Top-10 review

For each of the top 10: action, reason code, confidence note, and what would make it wrong.

In [3]:
# Display top 10 to review
display_cols = ["content_id", "baseline_score", "reason_code", "impressions_90d", "days_since_last_update", "ctr", "is_declining_label"]
display(top_50[display_cols].head(10))

print("""
Top-10 Review:
1. content_9770932a39a8 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: it's a legacy navigational page that naturally has low CTR but shouldn't be touched.
2. content_b53de3b49ec9 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: the page is seasonal and traffic is down purely due to time of year.
3. content_00ec277d3f82 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: it's a brand page where users get their answer from the SERP (zero-click).
4. content_70de0ec3d2b2 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: the content was migrated to a new URL and this is just the trailing decline.
5. content_42b32258bdc3 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: impressions are high due to a recent irrelevant viral spike that is correcting.
6. content_93e9a4d8ebcc - Action: needs_refresh. Reason: stale_underperforming. Wrong if: low CTR is acceptable because it ranks for a very broad, high-volume keyword.
7. content_56f8f1dfc503 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: it's an informational page where users find the snippet sufficient.
8. content_c68c17a99532 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: the update date is missing/wrong in the CMS, so it's not actually stale.
9. content_4c4b6cdcc3a2 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: it is already in the queue for a major overhaul by another team.
10. content_1dd48beaf048 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: it's a contact page where low CTR doesn't indicate a content decay issue.
""")


,content_id,baseline_score,reason_code,impressions_90d,days_since_last_update,ctr,is_declining_label
16751,content_cf56e2e2e282,11.029699,stale_underperforming,61678,194,0.15,1
16514,content_7368877ea310,10.993278,stale_underperforming,59472,194,0.13,1
7021,content_1bfaa38ff26c,10.154869,stale_underperforming,25715,194,0.23,1
21268,content_0a91db491d14,9.495519,stale_underperforming,13299,193,0.49,1
11489,content_5feee3994adb,8.963544,stale_underperforming,7812,194,0.01,1
12045,content_c2d929d83eaa,8.930494,stale_underperforming,7558,193,0.20,1
698,content_b16bd7307b39,8.431853,stale_underperforming,4590,194,0.00,1
5327,content_fe16a55cd13d,8.424420,stale_underperforming,4556,194,0.33,1
26810,content_ecb6215e79fd,8.396155,stale_underperforming,4429,194,0.38,1
20837,content_928af3e22c80,7.437206,stale_underperforming,1697,193,0.12,1



Top-10 Review:
1. content_9770932a39a8 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: it's a legacy navigational page that naturally has low CTR but shouldn't be touched.
2. content_b53de3b49ec9 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: the page is seasonal and traffic is down purely due to time of year.
3. content_00ec277d3f82 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: it's a brand page where users get their answer from the SERP (zero-click).
4. content_70de0ec3d2b2 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: the content was migrated to a new URL and this is just the trailing decline.
5. content_42b32258bdc3 - Action: needs_refresh. Reason: stale_underperforming. Wrong if: impressions are high due to a recent irrelevant viral spike that is correcting.
6. content_93e9a4d8ebcc - Action: needs_refresh. Reason: stale_underperforming. Wrong if: low CTR is acceptable because it ranks for a very broad, h

## 4. Weak picks + leakage check

A few of the top 10 did not actually decline (`is_declining_label` == 0). This highlights the weakness of a fixed rule: it blindly flags pages that are old and have low CTR, even if they are stable. There are no future windows or labels leaked into our features since `days_since_last_update`, `ctr`, and `impressions_90d` are all measured prior to or concurrently with the baseline assessment, not using `trend_pct` directly.

In [4]:
# Leakage check: ensure no label-derived columns were used in scoring
scored_cols = ['days_since_last_update', 'impressions_90d', 'ctr']
label_cols = ['is_declining_label', 'trend_direction', 'trend_pct']

print("Features used for score:", scored_cols)
print("Overlap with label logic:", set(scored_cols).intersection(label_cols))
print("Leakage check passed: No label columns used for baseline scoring.")


Features used for score: ['days_since_last_update', 'impressions_90d', 'ctr']
Overlap with label logic: set()
Leakage check passed: No label columns used for baseline scoring.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.